In [38]:

import numpy as np
import pandas as pd
import torch
import folium
from folium.plugins import MeasureControl
from ipywidgets import interact, IntSlider
from IPython.display import display

from lstm import lstm_seq2seq

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cuda


In [39]:

# =========================
# CONFIG
# =========================
TEST_CSV = "../data/vessel_prediction_model_data/mid_model_data/mid_test_data.csv"
MODEL_PATH = "best_lstm_seq2seq_dlat_dlon.pt"

feature_cols = [
    "LAT",
    "LON",
    "SPEED",
    "dt",
    "cog_sin",
    "cog_cos",
    "hdg_sin",
    "hdg_cos",
    "dlat",
    "dlon",
]

target_cols = ["dlat", "dlon"]

SEQ_LEN = 10
TARGET_LEN = 1

HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.2


In [40]:

# =========================
# LOAD RAW TEST DATA
# =========================
df_test = pd.read_csv(TEST_CSV)
df_test["TIME"] = pd.to_datetime(df_test["TIME"])
df_test.head()


,MMSI,LAT,LON,PERIOD,SPEED,COG,HEADING,voyage_id,TIME,dt,dlat,dlon,cog_rad,cog_sin,cog_cos,hdg_rad,hdg_sin,hdg_cos
0,205111000,22.697428,-78.498302,2023-09-01 14:10:00,13.2,108.4,107.0,1_205111000,2023-09-01 14:10:00,0.0,0.000000,0.000000,1.891937,0.948876,-0.315649,1.867502,0.956305,-0.292372
1,205111000,22.694939,-78.490204,2023-09-01 14:15:00,13.2,108.4,107.0,1_205111000,2023-09-01 14:15:00,300.0,-0.002489,0.008098,1.891937,0.948876,-0.315649,1.867502,0.956305,-0.292372
2,205111000,22.681163,-78.446551,2023-09-01 14:25:00,13.2,109.3,108.0,1_205111000,2023-09-01 14:25:00,600.0,-0.013776,0.043653,1.907645,0.943801,-0.330514,1.884956,0.951057,-0.309017
3,205111000,22.645493,-78.339608,2023-09-01 14:55:00,13.6,109.8,110.0,1_205111000,2023-09-01 14:55:00,1800.0,-0.035670,0.106943,1.916372,0.940881,-0.338738,1.919862,0.939693,-0.342020
4,205111000,22.637551,-78.316039,2023-09-01 15:00:00,13.7,108.9,110.0,1_205111000,2023-09-01 15:00:00,300.0,-0.007942,0.023569,1.900664,0.946085,-0.323917,1.919862,0.939693,-0.342020


In [41]:

# =========================
# WINDOWING (RAW VALUES)
# =========================
def make_windows_from_df(
    df: pd.DataFrame,
    feature_cols,
    target_cols,
    seq_len,
    target_len,
    route_col="voyage_id",
    time_col="TIME",
):
    X_list = []
    Y_list = []
    meta_rows = []

    work = df.copy()
    work[time_col] = pd.to_datetime(work[time_col])

    for route_id, g in work.groupby(route_col, sort=False):
        g = g.sort_values(time_col).reset_index(drop=True)

        feat = g[feature_cols].to_numpy(dtype=np.float32)
        targ = g[target_cols].to_numpy(dtype=np.float32)

        n = len(g)
        window_count = n - seq_len - target_len + 1
        if window_count <= 0:
            continue

        for i in range(window_count):
            x_win = feat[i : i + seq_len]
            y_win = targ[i + seq_len : i + seq_len + target_len]

            X_list.append(x_win)
            Y_list.append(y_win)

            meta_rows.append({
                "voyage_id": route_id,
                "start_idx": i,
                "history_end_idx": i + seq_len - 1,
                "target_start_idx": i + seq_len,
                "last_known_time": g.iloc[i + seq_len - 1][time_col],
                "target_time": g.iloc[i + seq_len][time_col],
            })

    X = np.stack(X_list, axis=0)  # (N, seq_len, n_features)
    Y = np.stack(Y_list, axis=0)  # (N, target_len, n_targets)

    X = np.transpose(X, (1, 0, 2))  # (seq_len, N, n_features)
    Y = np.transpose(Y, (1, 0, 2))  # (target_len, N, n_targets)

    return torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32), pd.DataFrame(meta_rows)


In [42]:

X_test, Y_test, meta_test = make_windows_from_df(
    df_test,
    feature_cols,
    target_cols,
    seq_len=SEQ_LEN,
    target_len=TARGET_LEN,
    route_col="voyage_id",
    time_col="TIME",
)

print("X_test shape:", tuple(X_test.shape))
print("Y_test shape:", tuple(Y_test.shape))
meta_test.head()


X_test shape: (10, 1538, 10)
Y_test shape: (1, 1538, 2)


,voyage_id,start_idx,history_end_idx,target_start_idx,last_known_time,target_time
0,5_209425000,0,9,10,2024-02-13 13:50:00,2024-02-13 13:55:00
1,5_209425000,1,10,11,2024-02-13 13:55:00,2024-02-13 14:00:00
2,5_209425000,2,11,12,2024-02-13 14:00:00,2024-02-13 14:05:00
3,5_209425000,3,12,13,2024-02-13 14:05:00,2024-02-13 14:15:00
4,5_209425000,4,13,14,2024-02-13 14:15:00,2024-02-13 14:20:00


In [43]:

# =========================
# LOAD MODEL
# =========================
model = lstm_seq2seq(
    input_size=len(feature_cols),
    hidden_size=HIDDEN_SIZE,
    output_size=2,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    target_indices=(0, 1),
    decoder_feature_indices=tuple(range(len(feature_cols))),
    predict_deltas=True,
).to(DEVICE)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("Model loaded from:", MODEL_PATH)


Model loaded from: best_lstm_seq2seq_dlat_dlon.pt


C:\Users\logan\AppData\Local\Temp\ipykernel_22100\2934980311.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=D

In [44]:

# =========================
# PREDICT RAW DELTAS
# =========================
with torch.no_grad():
    Y_pred = model.predict(
        X_test.to(DEVICE),
        target_len=TARGET_LEN,
        return_absolute=False,
    )

print("Y_pred shape:", tuple(Y_pred.shape))


Y_pred shape: (1, 1538, 2)


In [45]:

# =========================
# HELPERS
# =========================
YARDS_PER_METER = 1.0936132983377078

def deltas_to_absolute(start_points, deltas):
    """
    start_points: (N, 2)
    deltas: (target_len, N, 2)
    returns: (target_len, N, 2)
    """
    current = start_points.copy()
    out = []

    for t in range(deltas.shape[0]):
        current = current + deltas[t]
        out.append(current.copy())

    return np.stack(out, axis=0)

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2.0 * np.arcsin(np.sqrt(a))
    return R * c

def metrics_yards(pred_abs, true_abs):
    err_m = haversine_m(
        true_abs[..., 0],
        true_abs[..., 1],
        pred_abs[..., 0],
        pred_abs[..., 1],
    )
    err_yd = err_m * YARDS_PER_METER

    mae_yd = np.mean(np.abs(err_yd))
    mse_yd2 = np.mean(err_yd ** 2)
    rmse_yd = np.sqrt(mse_yd2)
    return mae_yd, mse_yd2, rmse_yd, err_yd


In [46]:

# =========================
# REBUILD ABSOLUTE POSITIONS
# =========================
X_test_np = X_test.detach().cpu().numpy().copy()
Y_test_np = Y_test.detach().cpu().numpy().copy()
Y_pred_np = Y_pred.copy()

# Last observed absolute lat/lon from history window
history_abs_all = X_test_np[:, :, [feature_cols.index("LAT"), feature_cols.index("LON")]].copy()
history_last_abs = history_abs_all[-1, :, :]   # (N, 2)

true_future_abs = deltas_to_absolute(history_last_abs, Y_test_np)
pred_future_abs = deltas_to_absolute(history_last_abs, Y_pred_np)

print("history_last_abs[0]:", history_last_abs[0])
print("true_future_abs[0,0]:", true_future_abs[0, 0])
print("pred_future_abs[0,0]:", pred_future_abs[0, 0])


history_last_abs[0]: [ 21.78906  -76.962166]
true_future_abs[0,0]: [ 21.798918 -76.9864  ]
pred_future_abs[0,0]: [ 21.802576 -76.99076 ]


In [47]:

# =========================
# LSTM METRICS IN YARDS
# =========================
mae_yd, mse_yd2, rmse_yd, err_yd = metrics_yards(pred_future_abs, true_future_abs)

print("Saved LSTM metrics on raw test data")
print(f"MAE  (yd): {mae_yd:,.3f}")
print(f"MSE  (yd²): {mse_yd2:,.3f}")
print(f"RMSE (yd): {rmse_yd:,.3f}")


Saved LSTM metrics on raw test data
MAE  (yd): 1,406.391
MSE  (yd²): 5,597,282.000
RMSE (yd): 2,365.858


In [48]:

# =========================
# OPTIONAL: SIMPLE DEAD RECKONING BASELINE
# Uses last observed SPEED + COG and the target-row dt.
# =========================
def dead_reckoning_next_point(df, meta_df, feature_cols):
    out = np.zeros((len(meta_df), 2), dtype=np.float64)

    for sample_idx, row in meta_df.iterrows():
        route_id = row["voyage_id"]
        h_idx = int(row["history_end_idx"])
        t_idx = int(row["target_start_idx"])

        g = df[df["voyage_id"] == route_id].sort_values("TIME").reset_index(drop=True)

        last_lat = float(g.loc[h_idx, "LAT"])
        last_lon = float(g.loc[h_idx, "LON"])
        speed_knots = float(g.loc[h_idx, "SPEED"])

        # Prefer target-row dt when available
        dt_seconds = float(g.loc[t_idx, "dt"])

        if "COG" in g.columns:
            cog_deg = float(g.loc[h_idx, "COG"])
        else:
            cog_deg = np.degrees(np.arctan2(float(g.loc[h_idx, "cog_sin"]), float(g.loc[h_idx, "cog_cos"]))) % 360

        speed_mps = speed_knots * 0.514444
        distance_m = speed_mps * dt_seconds

        bearing = np.radians(cog_deg)
        d_north_m = distance_m * np.cos(bearing)
        d_east_m = distance_m * np.sin(bearing)

        dlat = d_north_m / 111320.0
        dlon = d_east_m / (111320.0 * np.cos(np.radians(last_lat)))

        out[sample_idx, 0] = last_lat + dlat
        out[sample_idx, 1] = last_lon + dlon

    return out

dr_next_abs = dead_reckoning_next_point(df_test, meta_test, feature_cols)
dr_future_abs = dr_next_abs.reshape(1, dr_next_abs.shape[0], 2)

mae_yd_dr, mse_yd2_dr, rmse_yd_dr, err_yd_dr = metrics_yards(dr_future_abs, true_future_abs)

print("Dead reckoning baseline metrics")
print(f"MAE  (yd): {mae_yd_dr:,.3f}")
print(f"MSE  (yd²): {mse_yd2_dr:,.3f}")
print(f"RMSE (yd): {rmse_yd_dr:,.3f}")


Dead reckoning baseline metrics
MAE  (yd): 531.737
MSE  (yd²): 927,777.321
RMSE (yd): 963.212


In [49]:

# =========================
# MAP A SINGLE SAMPLE
# =========================
def map_single_prediction(sample_idx):
    history_abs = history_abs_all[:, sample_idx, :]
    pred_abs = pred_future_abs[:, sample_idx, :]
    true_abs = true_future_abs[:, sample_idx, :]

    voyage_id = meta_test.iloc[sample_idx]["voyage_id"]
    last_time = meta_test.iloc[sample_idx]["last_known_time"]
    target_time = meta_test.iloc[sample_idx]["target_time"]

    center_lat = np.mean([history_abs[-1, 0], pred_abs[0, 0], true_abs[0, 0]])
    center_lon = np.mean([history_abs[-1, 1], pred_abs[0, 1], true_abs[0, 1]])

    m = folium.Map(location=[center_lat, center_lon], zoom_start=9)
    m.add_child(MeasureControl(primary_length_unit="miles", secondary_length_unit="yards"))

    history_coords = [(row[0], row[1]) for row in history_abs]
    pred_coords = [(row[0], row[1]) for row in pred_abs]
    true_coords = [(row[0], row[1]) for row in true_abs]

    folium.PolyLine(history_coords, weight=3, tooltip="History").add_to(m)
    folium.PolyLine(pred_coords, weight=3, dash_array="5, 8", color="red", tooltip="Predicted").add_to(m)
    folium.PolyLine(true_coords, weight=3, dash_array="5, 8", color="green", tooltip="Actual").add_to(m)

    folium.CircleMarker(
        location=history_coords[-1],
        radius=7,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.9,
        tooltip="Last known AIS",
        popup=f"Voyage: {voyage_id}<br>Last known: {last_time}<br>Lat: {history_coords[-1][0]:.6f}<br>Lon: {history_coords[-1][1]:.6f}",
    ).add_to(m)

    folium.CircleMarker(
        location=pred_coords[0],
        radius=7,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.9,
        tooltip="Predicted next point",
        popup=f"Target time: {target_time}<br>Pred lat: {pred_coords[0][0]:.6f}<br>Pred lon: {pred_coords[0][1]:.6f}",
    ).add_to(m)

    folium.CircleMarker(
        location=true_coords[0],
        radius=7,
        color="green",
        fill=True,
        fill_color="green",
        fill_opacity=0.9,
        tooltip="Actual next point",
        popup=f"Target time: {target_time}<br>True lat: {true_coords[0][0]:.6f}<br>True lon: {true_coords[0][1]:.6f}",
    ).add_to(m)

    return m

def show_prediction(sample_idx):
    print(f"Sample index: {sample_idx}")
    print("Voyage:", meta_test.iloc[sample_idx]["voyage_id"])
    print("Last known time:", meta_test.iloc[sample_idx]["last_known_time"])
    print("Target time:", meta_test.iloc[sample_idx]["target_time"])
    display(map_single_prediction(sample_idx))

interact(
    show_prediction,
    sample_idx=IntSlider(min=0, max=X_test.shape[1] - 1, step=1, value=0)
)


interactive(children=(IntSlider(value=0, description='sample_idx', max=1537), Output()), _dom_classes=('widget…

<function __main__.show_prediction(sample_idx)>